# 新增流速因子：第一轮评测

16个因子，13币，每日固定01:00–02:00 UTC。训练09-19至24、验证09-25至27、测试09-28至10-04。全部按供应商接收时间，参数只用更早09-18小时数据。

这是探索性结果：固定一小时不能代表全天；测试仅7个共同日期块且历史已被研究。对照是同钟基础因子，不是原50因子模型，目标不是条件成交后Markout。未执行交易策略、未部署。

## 数据与结果来源

运行命令：`python tools/evaluate_flow_factors.py`。协议在doc/factors/flow-v1/evaluation/PROTOCOL.md，原数据在data/cryptohftdata。以下只读已保存结果，不重新联网/训练。

首轮结论：近端流组对5s收益预测MSE改善约0.129%，全部16约0.145%，幅度很小；量时钟方向增量接近0。30s波动模型全部16改善约1.164%，但缺失指示单独也改善约0.897%，不能将全部改善归因于因子数值。尚未通过原50同钟或经济采用闸门。

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/factor-library/evaluation-v1/results.json').exists())
r=json.loads((root/'data/factor-library/evaluation-v1/results.json').read_text())
print('训练/验证/测试行数：',r['rows'])
print('原始小时：',r['source_hours'],'；拒绝小时：',r['rejected_hours'])
print('范围：',r['scope'])
plt.rcParams.update({'figure.figsize':(10,5),'font.size':11,'axes.spines.top':False,'axes.spines.right':False})


## 因子覆盖与独立相关性
分别报告两个目标。IC为每个币每天的双变量Spearman，再按共同日期等权汇总；不能把所有秒行当独立样本。BH q仅为小样本描述性校正，不是交易采用证据。

In [ ]:
factor_rows=[]
for x in r['single_factors']:
    stats=x['test_daily_equal_coin_ic']
    factor_rows.append({'factor':x['factor'],'target':x['target'],'coverage':x['test_coverage'],
                       'daily_IC':stats['mean'],'CI_low':stats['ci95'][0],'CI_high':stats['ci95'][1],
                       'calendar_days':stats['days'],'BH_q_descriptive':x['bh_q_descriptive']})
factors=pd.DataFrame(factor_rows)
for target,g in factors.groupby('target',sort=False):
    print(target)
    display(g.round(4).reset_index(drop=True))


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,7),sharey=True)
for ax,target in zip(axes,['return_5s_bp','future_rv_30s_bp']):
    g=factors[factors.target==target].sort_values('factor')
    x=g.daily_IC.to_numpy(float);lo=g.CI_low.to_numpy(float);hi=g.CI_high.to_numpy(float)
    pos=np.arange(len(g));ax.hlines(pos,lo,hi,color='#596579');ax.scatter(x,pos,color='#24756b',s=25)
    ax.axvline(0,color='#333',linewidth=.8);ax.set_yticks(pos,g.factor);ax.set(title=target,xlabel='Equal-coin daily Spearman IC')
axes[0].invert_yaxis()
fig.suptitle('Test: 2026-09-28 to 10-04 | 7 shared date blocks',fontsize=12)
fig.tight_layout();fig.savefig(root/'data/factor-library/evaluation-v1/factor_ic.png',dpi=140);plt.show()


## 各因子组相对基础对照的增量
正值表示配对测试MSE降低，负值表示变差。日历块区间跨0时不能声称改善稳定；统计改善也不等于交易收益。所有组合固定参数，同一目标使用相同标签可用行，模型原生处理NaN。

In [ ]:
models=pd.DataFrame([{'target':x['target'],'arm':x['arm'],'MSE_gain_percent':x['mse_improvement_percent'],
                     'CI_low':x['mse_improvement_ci95'][0],'CI_high':x['mse_improvement_ci95'][1],
                     'test_equal_coin_day_MSE':x['test_equal_coin_day_mse'],'test_rows':x['test_rows']}
                    for x in r['model_arms']])
display(models.round(4))
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,target in zip(axes,['return_5s_bp','future_rv_30s_bp']):
    g=models[(models.target==target)&(models.arm!='base')]
    pos=np.arange(len(g));ax.hlines(pos,g.CI_low,g.CI_high,color='#596579');ax.scatter(g.MSE_gain_percent,pos,color='#24756b',s=30)
    ax.axvline(0,color='#333',linewidth=.8);ax.set_yticks(pos,g.arm);ax.invert_yaxis()
    ax.set(title=target,xlabel='Paired MSE improvement (%)')
fig.tight_layout();fig.savefig(root/'data/factor-library/evaluation-v1/model_gain.png',dpi=140);plt.show()


## 币对差异
跨币共用日期块，不把13币当13个独立显著性投票。以下展示全部16一起加入时，每币相对于基础对照的误差变化。

In [ ]:
for arm in r['model_arms']:
    if arm['arm']=='all16':
        print(arm['target'])
        display(pd.DataFrame(arm['per_coin']).T.round(4))


## 判断边界与下一步

保留覆盖率、区间和未通过组合，不仅展示最好结果。成交额桶有值的时段本身可能带有活跃度信息，原生缺失分支增量不独立证明桶内数值有效。

下一步对有研究价值的候选冻结更小集合，重建原50因子的同接收时钟基线，做增量与缺失指示消融，再验证全天、更长独立日期、延迟衰减以及带排队和费用的经济结果。当前不更新adopted状态。

## 后验敏感性诊断

以下在主结果之后追加，不是事前确认性检验，也未按最好随机种子选模型。missing_only只添加16个缺失指示；它不能与all16简单相减来分解因子数值贡献。

In [ ]:
s=json.loads((root/'data/factor-library/evaluation-v1/sensitivity.json').read_text())
display(pd.DataFrame([{'target':x['target'],'seed':x['seed'],'arm':x['arm'],
                       'MSE_gain_percent':x['improvement_percent']} for x in s['rows']]).round(4))
